# P03 — Memoria larga de corto plazo

## 1. Título y paper

**Paper:** *Long Short-Term Memory*  
**Autoría:** Sepp Hochreiter, Jürgen Schmidhuber  
**Año y venue:** 1997 · Neural Computation, 9(8), 1735–1780  
**Nivel:** L2 · **Motor:** `lstm`  
**Ficha completa:** [`P03_lstm`](../../papers/foundational/P03_lstm/README.md)

**Hito:** Primera arquitectura recurrente capaz de mantener información a través de cientos de pasos sin que el gradiente se desvanezca.

- [DOI (Neural Computation)](https://doi.org/10.1162/neco.1997.9.8.1735)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: En un RNN el gradiente se multiplica en cada paso temporal: se desvanece o explota, y la red no aprende dependencias largas.
2. Ejecutar una implementación mínima de la propuesta: Una celda con estado aditivo (carrusel de error constante) y puertas multiplicativas que deciden qué entra y qué sale.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P02
- Elman (1990), redes recurrentes simples
- Hochreiter (1991), diagnóstico del gradiente desvaneciente


## 4. Intuición

Una cinta transportadora que atraviesa el tiempo sin ser tocada, y tres compuertas que deciden qué se sube, qué se baja y qué se mira. El truco no son las compuertas: es que la cinta se actualiza **sumando**, no multiplicando.


## 5. Concepto mínimo

```text
f = σ(W_f·[h, x])   olvido      c = f ⊙ c_prev + i ⊙ g     ← suma, no producto encadenado
i = σ(W_i·[h, x])   entrada     h = o ⊙ tanh(c)
o = σ(W_o·[h, x])   salida
g = tanh(W_g·[h, x]) candidato
```

En un RNN clásico el gradiente se multiplica por `W·tanh'` en cada paso: si ese factor es 0,4, en 40 pasos queda `0,4⁴⁰ ≈ 1e-16`. En la celda, `∂c_t/∂c_{t-1} = f ≈ 1`.


## 6. Código explicado

El motor calcula ambos decaimientos y una pasada completa de la celda con valores explícitos.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('lstm', seed=7)['result']
show(r['gates'])
show(r['gradient_after_40_steps'])

## 7. Predicción antes de ejecutar

1. Tras 40 pasos, ¿cuántos órdenes de magnitud separan el gradiente del RNN del de la celda?
2. Si la puerta de olvido valiera 0,5 en lugar de ~1, ¿la celda seguiría preservando el gradiente?
3. ¿Qué puerta controla lo que el resto de la red *ve*, sin borrar lo que la celda *recuerda*?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
import math

for f in (1.00, 0.98, 0.90, 0.50):
    grad = 1.0
    for _ in range(40):
        grad *= f
    print(f'puerta de olvido={f:.2f} → gradiente tras 40 pasos = {grad:.3e}')

## 9. Salida interpretable

Con `f = 1,00` el gradiente se conserva exacto (carrusel de error constante). Con `f = 0,50` la celda vuelve a desvanecerse: **la LSTM no elimina el problema, lo pone bajo control de una puerta aprendida**. Esa distinción es la respuesta correcta en un examen.


## 10. Comentario pedagógico

Ojo con el anacronismo: el paper de 1997 tenía puertas de entrada y salida. La puerta de olvido —la que acabas de manipular— la añadieron Gers, Schmidhuber y Cummins en 1999/2000. Atribuirla al paper original es un error frecuente en resúmenes de internet.


## 11. Error o anti-patrón deliberado

Anti-patrón: explicar la LSTM diciendo «resuelve el gradiente desvaneciente» sin condición alguna.


In [ ]:
afirmacion = 'La LSTM resuelve el gradiente desvaneciente.'
print(afirmacion)
print('→ falso como enunciado absoluto: acabas de ver f=0.50 desvanecerse igual.')

## 12. Corrección

Enunciado correcto, con su condición explícita:


In [ ]:
correcto = ('La LSTM MITIGA el gradiente desvaneciente cuando la puerta de olvido '
            'aprende a mantenerse cerca de 1 en el intervalo que hay que recordar.')
print(correcto)

## 13. Desafío guiado

¿A partir de qué valor de `f` el gradiente cae por debajo de 1e-3 en 40 pasos? Búscalo numéricamente.


In [ ]:
umbral = 1e-3
f = 1.0
while f > 0:
    if f ** 40 < umbral:
        break
    f -= 0.005
print(f'la puerta debe mantenerse por encima de f≈{f + 0.005:.3f} para conservar 1e-3 en 40 pasos')

## 14. Desafío autónomo

Implementa la tarea de «copia con retardo»: la red debe repetir un símbolo visto T pasos antes. Mide la precisión con un RNN tanh y con una celda LSTM para T = 5, 20 y 100. Reporta la semilla y el número de parámetros de cada modelo.


## 15. Evidencia de aprendizaje

Guarda la tabla de decaimiento por valor de puerta, el umbral que encontraste y el enunciado corregido sobre qué resuelve y qué no resuelve la LSTM.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P03_lstm/README.md) · evaluación formal: [`assessments/papers/P03_lstm.md`](../../assessments/papers/P03_lstm.md)


## 16. Cierre

Ya se pueden modelar secuencias largas. Falta que la entrada y la salida puedan tener longitudes distintas, que es lo que exige traducir.


## 17. Conexión con el siguiente hito

- P06

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
